In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

CATALOG     = "cinedata_analytics"
SCHEMA      = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_inputs/"

# Mapeamento arquivo -> tabela Bronze.
# O arquivo de info usa wildcard porque o nome físico recebido (movies_info_TMDB_IMDB.csv)
# diverge do nome citado no escopo (movies_info_IMDB_TMDB.csv) — divergência validada com o orientador.
ARQUIVOS_TABELAS = {
    "movies_info*.csv":                "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":    "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":  "tb_credits_and_tags",
    "movies_reviews.csv":              "tb_movies_reviews",
}

# Arquivo com a resposta da API PTAX, salvo manualmente no Volume
ARQUIVO_COTACAO = f"{VOLUME_PATH}cotacao_dolar.json"

In [0]:
# IF NOT EXISTS torna o notebook idempotente: pode ser executado várias vezes pelo Job sem erro.
# Obs.: o catálogo e o Volume já precisam existir (criados manualmente antes do upload dos arquivos).
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA} COMMENT 'Camada Bronze - dados brutos, sem tratamento'")
spark.sql(f"USE CATALOG {CATALOG}")

DataFrame[]

In [0]:
# Decisões de leitura:
# - inferSchema=False -> tudo STRING. Os dados têm sujeiras intencionais ("154,34", "$ 97000000",
#   "Unknown", textos deslocados por column shift). Inferir tipos transformaria esses valores em NULL
#   já na Bronze, violando a regra "sem alteração de conteúdo". A tipagem fica na Silver.
# - multiLine=True -> sinopses/taglines têm quebras de linha dentro de campos entre aspas.
# - quote='"' + escape='"' -> os arquivos seguem o padrão RFC 4180 (aspas internas escritas como "").
#   Com o escape padrão do Spark (\), o movies_info perdia ~5 mil registros (101.571 vs 106.596).
# - mode=PERMISSIVE -> linhas malformadas (column shift) não interrompem o pipeline; são tratadas na Silver.
CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
    "multiLine": "true",
    "quote": '"',
    "escape": '"',
    "mode": "PERMISSIVE",
    "encoding": "UTF-8",
}

def ingerir_csv_bronze(padrao_arquivo: str, tabela: str) -> None:
    """Lê um CSV do Volume, adiciona ingestion_datetime e faz append em Delta na Bronze."""
    caminho = f"{VOLUME_PATH}{padrao_arquivo}"

    df = spark.read.options(**CSV_OPTIONS).csv(caminho)

    # Coluna de auditoria: momento exato da inserção na Bronze.
    # A Silver usa essa coluna para manter apenas a versão mais recente de cada registro.
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    (df.write
       .format("delta")
       .mode("append")                   # requisito: histórico de cargas, nunca sobrescreve
       .option("mergeSchema", "false")   # a estrutura da Bronze não deve mudar entre cargas
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{tabela}"))

    print(f"OK  {padrao_arquivo:35s} -> {CATALOG}.{SCHEMA}.{tabela}  ({df.count():,} linhas nesta carga)")

In [0]:
# Executa a ingestão dos 5 arquivos
for padrao, tabela in ARQUIVOS_TABELAS.items():
    ingerir_csv_bronze(padrao, tabela)

OK  movies_info*.csv                    -> cinedata_analytics.bronze.tb_movies_info  (106,596 linhas nesta carga)
OK  movies_financials_IMDB_TMDB.csv     -> cinedata_analytics.bronze.tb_movies_financials  (106,165 linhas nesta carga)
OK  movies_metrics_IMDB_TMDB.csv        -> cinedata_analytics.bronze.tb_movies_metrics  (103,072 linhas nesta carga)
OK  credits_and_tags_IMDB_TMDB.csv      -> cinedata_analytics.bronze.tb_credits_and_tags  (105,608 linhas nesta carga)
OK  movies_reviews.csv                  -> cinedata_analytics.bronze.tb_movies_reviews  (32,412 linhas nesta carga)


In [0]:
# Ingestão da cotação do dólar (PTAX / Banco Central)
# CONTEXTO: o Databricks Free Edition (Serverless) restringe o acesso de saída à internet,
# então a chamada direta à API não é possível neste ambiente. A resposta da API foi obtida
# manualmente pelo navegador, com o endpoint oficial abaixo, e salva em cotacao_dolar.json no Volume:
#
# https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(
#   dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'
#   &@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json
#
# Os widgets documentam o período consultado (formato MM-DD-AAAA, exigido pela API)
# e são usados só para conferir o conteúdo do arquivo.
dbutils.widgets.text("data_inicio", "", "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim",    "", "Data fim (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio").strip()
data_fim    = dbutils.widgets.get("data_fim").strip()
print(f"Período de referência informado: {data_inicio or '(não informado)'} até {data_fim or '(não informado)'}")

Período de referência informado: (não informado) até (não informado)


In [0]:
# Schema explícito espelhando a resposta OData da API:
# {"@odata.context": "...", "value": [{"cotacaoCompra": ..., "dataHoraCotacao": "..."}]}
# Com o schema definido, a estrutura da tabela fica garantida e o Spark não precisa inferir tipos.
schema_resposta_api = StructType([
    StructField("@odata.context", StringType(), True),
    StructField("value", ArrayType(StructType([
        StructField("cotacaoCompra",   DoubleType(), True),
        StructField("dataHoraCotacao", StringType(), True),
    ])), True),
])

# multiLine=True: o JSON é um único objeto (não JSON Lines), possivelmente formatado em várias linhas.
df_json = (spark.read
                .schema(schema_resposta_api)
                .option("multiLine", "true")
                .json(ARQUIVO_COTACAO))

# Um arquivo corrompido ou com formato diferente faz "value" vir nulo.
# Nesse caso o notebook falha explicitamente em vez de seguir sem dados.
if df_json.filter(F.col("value").isNotNull()).isEmpty():
    raise ValueError(f"O arquivo {ARQUIVO_COTACAO} não contém o array 'value' esperado da API PTAX.")

In [0]:
# inline: "explode" o array de structs "value" e já promove os campos a colunas
# (equivale a explode + select v.campo, mas sem a coluna intermediária).
# Os campos ficam como vieram da API, como exige a Bronze.
# Estrutura da tabela: dataHoraCotacao STRING, cotacaoCompra DOUBLE, ingestion_datetime TIMESTAMP.
df_cotacao = (df_json
              .selectExpr("inline(value)")
              .selectExpr("dataHoraCotacao", "cotacaoCompra")
              .withColumn("ingestion_datetime", F.current_timestamp()))

# Conferência do período realmente presente no arquivo (compare com os widgets)
display(df_cotacao.agg(
    F.min("dataHoraCotacao").alias("primeira_cotacao"),
    F.max("dataHoraCotacao").alias("ultima_cotacao"),
    F.count("*").alias("qtd_cotacoes"),
))

primeira_cotacao,ultima_cotacao,qtd_cotacoes
2026-09-11 13:07:22.532196,2026-09-17 13:03:21.858212,5


In [0]:
if df_cotacao.isEmpty():
    # Array "value" vazio: o período consultado tinha só feriados/fins de semana.
    # Não grava carga vazia; a Silver reaproveita as cotações já existentes.
    print("AVISO: nenhuma cotação no arquivo. Nada foi gravado em tb_cotacao_dolar.")
else:
    (df_cotacao.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.tb_cotacao_dolar"))
    print(f"OK  cotacao_dolar.json -> {CATALOG}.{SCHEMA}.tb_cotacao_dolar  ({df_cotacao.count()} linhas nesta carga)")
    display(df_cotacao)

OK  cotacao_dolar.json -> cinedata_analytics.bronze.tb_cotacao_dolar  (5 linhas nesta carga)


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-11 13:07:22.532196,5.0912,2026-09-18T16:05:25.180Z
2026-09-14 13:10:08.144425,5.169,2026-09-18T16:05:25.180Z
2026-09-15 13:09:19.199664,5.1484,2026-09-18T16:05:25.180Z
2026-09-16 13:05:30.35873,5.152,2026-09-18T16:05:25.180Z
2026-09-17 13:03:21.858212,5.1515,2026-09-18T16:05:25.180Z


In [0]:
# Conferência: total acumulado e quantidade de cargas (execuções) em cada tabela Bronze.
tabelas = list(ARQUIVOS_TABELAS.values()) + ["tb_cotacao_dolar"]

resumo = []
for t in tabelas:
    nome = f"{CATALOG}.{SCHEMA}.{t}"
    if spark.catalog.tableExists(nome):
        df_t = spark.table(nome)
        resumo.append((
            t,
            df_t.count(),
            df_t.select("ingestion_datetime").distinct().count(),
            str(df_t.agg(F.max("ingestion_datetime")).first()[0]),
        ))

display(spark.createDataFrame(resumo, ["tabela", "total_linhas", "qtd_cargas", "ultima_ingestao"]))

tabela,total_linhas,qtd_cargas,ultima_ingestao
tb_movies_info,639576,6,2026-09-18 16:04:47.071532
tb_movies_financials,636990,6,2026-09-18 16:04:57.680068
tb_movies_metrics,618432,6,2026-09-18 16:05:02.187289
tb_credits_and_tags,633648,6,2026-09-18 16:05:06.598634
tb_movies_reviews,194472,6,2026-09-18 16:05:11.206360
tb_cotacao_dolar,25,5,2026-09-18 16:05:23.256290
